title: Geodata Visualisation\
summary: shapes, geometries, geospatial data formats, coordinate systems, grouped islands for statistical information, iris \
main reference: [GeoPandas](https://geopandas.org/en/stable/docs.html), [mapclassify](https://pypi.org/project/mapclassify/)\
scope: Data Science for Public Policy Course\
last update: 2025-04-22\
author: Noah Fröhlich\
revision: Silvia Tulli

# **Geodata Visualisation**

**Table of Contents**
* Geodata 101
* Environment Set up
  * Mount Google Drive
  * Install libraries
* Data preprocessing
* Create the map
* Change reference systems
* References

# Geodata 101

Geodata, also known as spatial data, is information tied to specific locations on Earth.  Imagine points of interest, boundaries of countries, or even weather patterns – all these can be represented as geodata in Python! To work with this data type, let's explore some fundamentals:

**1. Shapes and Geometries**

Geodata doesn't just represent points. It can depict various shapes on a map:

- Points: Represent precise locations with a single X and Y coordinate (think: a landmark).

- Lines: Depict linear features like roads or rivers. Imagine connecting multiple points with a sequence of coordinates.

- Polygons: Define closed areas like countries, parks, or buildings. Picture connecting points to form a closed loop.

These shapes are often called "geometries" in Python.

**Geospatial Data Formats**

Geospatial data can be stored in various formats, each suited for different purposes. Common formats include:
- Shapefile (.shp): A popular vector data format comprising points, lines, and polygons along with associated attributes. Shapefiles consist of multiple files, including .shp for geometry, .shx for index, and .dbf for attribute data.
- GeoJSON (.geojson): A lightweight format for encoding geospatial data structures using JSON.
- GeoTIFF (.tif): A raster format for storing georeferenced images, often used for satellite imagery and elevation data.
- KML/KMZ: Keyhole Markup Language, used for annotating and visualizing geographic information in Google Earth.

**Coordinate Systems:**

Coordinate systems are used to define the position of geographic features on the Earth's surface.

The two primary types of coordinate systems are:
- Geographic Coordinate System (GCS): Uses latitude and longitude to specify locations on a spherical or ellipsoidal model of the Earth.
- Projected Coordinate System (PCS): Utilizes x, y Cartesian coordinates on a flat surface (e.g., a map) to represent locations.

Common coordinate reference systems include:
- WGS84 (World Geodetic System 1984): A widely used GCS based on the Earth's ellipsoid model, commonly used for GPS coordinates.
- Web Mercator: A projected coordinate system used by many web mapping services like Google Maps and OpenStreetMap.
- Universal Transverse Mercator (UTM): A PCS divided into zones, commonly used for mapping specific regions.

When working with geodata from multiple sources, pay attention that they are stored in the coordinate system and projection. You may have to convert them to a common system before being able to work with them.

# Code examples

In this example, we'll look at the median household income in Paris. Our unit of analysis is a so-called IRIS. IRIS is an abbreviation for “Grouped Islands for Statistical Information”, which creates areas of roughly equal amounts of inhabitants. This provides a more granular analysis than looking at Arrondissements.

For the analysis, we'll work with two datasets. One provides information about the names and, crucially, the coordinates of the IRIS areas. Another dataset includes the income per IRIS. By merging those datasets, we can neatly map the information.

This colab shows only one way to create a map and a very simple one at that. So feel free to explore more functions and libraries and play around a bit with your data!

In [ ]:
# mount gdrive
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


In [ ]:
# install libraries
!pip install geopandas # geopandas is a useful library to work with geodata
!pip install mapclassify
import pandas as pd # classic
import geopandas as gpd # import geopandas (import both pandas and geopandas as short-forms to save some typing later)
import matplotlib.pyplot as plt # to vosualise some of the results
from shapely.geometry import Point, Polygon, LineString # useful for work with the geodata
from geopy.geocoders import Nominatim # also for geodata
import mapclassify # geodata once again
geolocator = Nominatim(user_agent = "geoapiExercises") # more geodata functions, yayy

   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 59.1/59.1 kB 2.3 MB/s eta 0:00:00


Note that not all librarires are needed for this colab, but feel free to check them out if you're looking for additional functionalities.

Please find both datasets [here](https://drive.google.com/drive/folders/1pkZBU4Ebw-9z7Be5KmncNmC1nvLPfX63?usp=drive_link).

In [ ]:
# load datasets
gdf_iris = gpd.read_file("/content/drive/MyDrive/05 TA Data Science/01 Spring Semester 2024/Map_data/IRIS-GE_2-0_SHP_LAMB93_D075-2022/IRIS_GE.SHP") # note that we load this df with geopandas to create a geopandas dataframe instead of a regular df
income = pd.read_csv("/content/drive/MyDrive/05 TA Data Science/01 Spring Semester 2024/Map_data/income.csv", delimiter = ";") # since this dataset does not include geodata we import with the regular pandas library

In the geometry column, you see that the shape of the IRIS is stored as polygon with multiple coordinates:

In [ ]:
# show income df
gdf_iris.head()

,INSEE_COM,NOM_COM,IRIS,CODE_IRIS,NOM_IRIS,TYP_IRIS,geometry
0,75116,Paris 16e Arrondissement,6110,751166110,Auteuil 10,H,"POLYGON ((645595.8 6860843.1, 645596 6860873.2..."
1,75116,Paris 16e Arrondissement,6129,751166129,Auteuil 29,H,"POLYGON ((646617.6 6861093.2, 646621.4 6861099..."
2,75115,Paris 15e Arrondissement,6006,751156006,Javel 6,H,"POLYGON ((647311.3 6860097.6, 647317.1 6860119..."
3,75115,Paris 15e Arrondissement,5712,751155712,Saint-Lambert 12,H,"POLYGON ((647843.9 6860072.7, 647851.2 6860089..."
4,75115,Paris 15e Arrondissement,5910,751155910,Grenelle 10,H,"POLYGON ((647965.2 6861229.2, 648005 6861296.8..."


## Data preprocessing

Before analysing the data, we have to brush it up a bit.

In [ ]:
# change the type of the CODE_IRIS column in gdf_iris from string to integer
gdf_iris['CODE_IRIS'] = gdf_iris['CODE_IRIS'].astype('int')

# keep only the median income column and the IRIS code (for clarity)
income_median = income[["IRIS", "mediane"]]

# add the median household income per iris from the income df to the gdf_iris (based on the IRIS number as common identifier)
gdf_iris = pd.merge(gdf_iris, income_median, left_on = 'CODE_IRIS', right_on = 'IRIS', how = 'outer')

# clean up the merged gdf a bit
gdf_iris = gdf_iris.drop("IRIS_y", axis=1) # drop the IRIS column because we already have the CODE_IRIS column with the same information
gdf_iris.rename(columns={'mediane':'median_income'}, inplace=True) # rename the median column to make it clearer
gdf_iris.rename(columns={'IRIS_x':'IRIS'}, inplace=True) # drop the x from the IRIS column
gdf_iris['median_income'] = gdf_iris['median_income'].astype(float) # make sure the median income column is float (because there are NAs in the column)
gdf_iris = gdf_iris.set_index("CODE_IRIS") # set the index to make the visualisation easier to read later

We now have a quite tidy gdf with all the relevant information: departement, name, IRIS, type of IRIS, spatial data, and median income per IRIS.

In [ ]:
gdf_iris.head(20)

,INSEE_COM,NOM_COM,IRIS,NOM_IRIS,TYP_IRIS,geometry,median_income
CODE_IRIS,,,,,,,
751010101,75101,Paris 1er Arrondissement,0101,Saint-Germain l'Auxerrois 1,H,"POLYGON ((651771.6 6862243.4, 651786.1 6862271...",NaN
751010102,75101,Paris 1er Arrondissement,0102,Saint-Germain l'Auxerrois 2,A,"POLYGON ((651668.7 6862071.1, 651679 6862086.2...",NaN
751010103,75101,Paris 1er Arrondissement,0103,Saint-Germain l'Auxerrois 3,A,"POLYGON ((651565.9 6862309.2, 651585.7 6862366...",NaN
751010104,75101,Paris 1er Arrondissement,0104,Saint-Germain l'Auxerrois 4,A,"POLYGON ((650849.7 6862532.4, 650850.3 6862538...",NaN
751010105,75101,Paris 1er Arrondissement,0105,Tuileries,D,"POLYGON ((650221 6862849.5, 650230.4 6862868.5...",NaN
751010199,75101,Paris 1er Arrondissement,0199,Seine et Berges,D,"POLYGON ((650181.1 6862768.3, 650210.6 6862828...",NaN
751010201,75101,Paris 1er Arrondissement,0201,Les Halles 1,H,"POLYGON ((652049.4 6862421.1, 652052.7 6862438...",32590.0
751010202,75101,Paris 1er Arrondissement,0202,Les Halles 2,H,"POLYGON ((651514.6 6862585.4, 651515.1 6862586...",41100.0
751010203,75101,Paris 1er Arrondissement,0203,Les Halles 3,H,"POLYGON ((651516.3 6862666.5, 651520.7 6862680...",35070.0


## Create the map

Because our data is so tidy, creating an interactive map is as simple as that:

In [ ]:
gdf_iris.explore("median_income", legend = True)

## Change reference systems

When you're working wth a single coordinate file, you usually do not have to worry about the reference system because most functions will adapt automatically to your data. In case you want to merge data from two different sources or run into errors, you might need to dive a bit deeper however.

Best case you will find the reference system of your data somewhere in the documentation. The [documentation](https://geoservices.ign.fr/sites/default/files/2021-10/DC_Limites_IRIS_0.pdf) for the IRIS dataset has this information on p. 6 for example.

Should you be unable to find useful meta-data, you can identify the reference system with this command:

In [ ]:
# get meta information about our gdf_iris
gdf_iris.crs

<Projected CRS: IGNF:LAMB93>
Name: RGF93 Lambert 93
Axis Info [cartesian]:
- [east]: Easting (metre)
- [north]: Northing (metre)
Area of Use:
- undefined
Coordinate Operation:
- name: unnamed
- method: Lambert Conic Conformal (2SP)
Datum: Reseau Geodesique Francais 1993 v1
- Ellipsoid: GRS 1980
- Prime Meridian: Greenwich

We are interested in the information in the very first line:

    Projected CRS: IGNF:LAMB93

This means that our data is in the IGNF:LAMB93 format. If you want or need to change that, you can use the "to_crs" command:

In [ ]:
gdf_iris_wgs = gdf_iris.to_crs("wgs84") # we transform into wgs84, here you can specify basically all other possible projections

Let's look at the difference:

In [ ]:
gdf_iris.head()

,INSEE_COM,NOM_COM,IRIS,NOM_IRIS,TYP_IRIS,geometry,median_income
CODE_IRIS,,,,,,,
751010101,75101,Paris 1er Arrondissement,0101,Saint-Germain l'Auxerrois 1,H,"POLYGON ((651771.6 6862243.4, 651786.1 6862271...",NaN
751010102,75101,Paris 1er Arrondissement,0102,Saint-Germain l'Auxerrois 2,A,"POLYGON ((651668.7 6862071.1, 651679 6862086.2...",NaN
751010103,75101,Paris 1er Arrondissement,0103,Saint-Germain l'Auxerrois 3,A,"POLYGON ((651565.9 6862309.2, 651585.7 6862366...",NaN
751010104,75101,Paris 1er Arrondissement,0104,Saint-Germain l'Auxerrois 4,A,"POLYGON ((650849.7 6862532.4, 650850.3 6862538...",NaN
751010105,75101,Paris 1er Arrondissement,0105,Tuileries,D,"POLYGON ((650221 6862849.5, 650230.4 6862868.5...",NaN


In [ ]:
gdf_iris_wgs.head()

,INSEE_COM,NOM_COM,IRIS,NOM_IRIS,TYP_IRIS,geometry,median_income
CODE_IRIS,,,,,,,
751010101,75101,Paris 1er Arrondissement,0101,Saint-Germain l'Auxerrois 1,H,"POLYGON ((2.34267 48.85842, 2.34287 48.85868, ...",NaN
751010102,75101,Paris 1er Arrondissement,0102,Saint-Germain l'Auxerrois 2,A,"POLYGON ((2.34129 48.85686, 2.34143 48.857, 2....",NaN
751010103,75101,Paris 1er Arrondissement,0103,Saint-Germain l'Auxerrois 3,A,"POLYGON ((2.33986 48.859, 2.34012 48.85951, 2....",NaN
751010104,75101,Paris 1er Arrondissement,0104,Saint-Germain l'Auxerrois 4,A,"POLYGON ((2.33007 48.86095, 2.33008 48.861, 2....",NaN
751010105,75101,Paris 1er Arrondissement,0105,Tuileries,D,"POLYGON ((2.32147 48.86375, 2.32159 48.86392, ...",NaN


If you compare the numbers in the geometry column, you'll notice that they're completely different than before. Still, if we map the transformed data, the map looks exactly the same:

In [ ]:
gdf_iris_wgs.explore("median_income", legend = True)

# References

La description des Iris recalés sur les données BD TOPO® (précision 1 m) (2022). *République Française géoservices*. https://geoservices.ign.fr/irisge (last accessed Dec 23, 2022)

Revenus, pauvreté et niveau de vie en 2019 (Iris) (2022). *Insee.* https://www.insee.fr/fr/statistiques/6049648#consulter (last accessed Dec 23, 2022)